## Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import pickle

## Model and Dataset

LFCC feature matrices of shape `[180, 321]` are treated as single-channel 2D images and passed through a CNN.

In [ ]:
class AudioCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.bn1   = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn2   = nn.BatchNorm2d(32)
        self.pool  = nn.MaxPool2d(2)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))
        self.fc1   = nn.Linear(32 * 4 * 4, 1)

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.adaptive_pool(x)
        x = x.view(x.size(0), -1)
        return self.fc1(x)


class AudioDataset(Dataset):
    def __init__(self, features_df, labels_df):
        self.data = pd.merge(features_df, labels_df, on='uttid')

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x = self.data.iloc[idx]['features']
        y = torch.tensor(self.data.iloc[idx]['label'], dtype=torch.float32)
        return x, y

## Data Loading

The `CompatibilityUnpickler` handles a NumPy version mismatch that arises when `.pkl` files were serialized with an older NumPy and loaded under a newer one.

In [ ]:
class CompatibilityUnpickler(pickle.Unpickler):
    """Handles numpy.core -> numpy._core rename introduced in NumPy 2.x."""
    def find_class(self, module, name):
        if module in ['numpy._core.numeric', 'numpy.core.numeric']:
            import numpy
            return numpy.core.numeric._frombuffer
        return super().find_class(module, name)

def load_pickle_compat(path):
    with open(path, 'rb') as f:
        return CompatibilityUnpickler(f).load()


# Update these paths to point to your local data directory
TRAIN_FEATURES = 'data/train/features.pkl'
TRAIN_LABELS   = 'data/train/labels.pkl'

print("Loading data...")
train_feats = load_pickle_compat(TRAIN_FEATURES)
train_labs  = load_pickle_compat(TRAIN_LABELS)

train_dataset = AudioDataset(train_feats, train_labs)
train_loader  = DataLoader(train_dataset, batch_size=32, shuffle=True)

model     = AudioCNN()
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"Dataset size: {len(train_dataset)} samples")

## Training

In [ ]:
EPOCHS = 12

model.train()
for epoch in range(EPOCHS):
    total_loss = 0
    for x, y in train_loader:
        optimizer.zero_grad()
        pred = model(x).squeeze()
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1:02d}/{EPOCHS}  Loss: {total_loss/len(train_loader):.4f}")

## Inference

In [ ]:
TEST_FEATURES = 'data/test/features.pkl'
test_df = load_pickle_compat(TEST_FEATURES)

model.eval()
uttids, preds = [], []

with torch.no_grad():
    for i in range(len(test_df)):
        feat  = test_df.iloc[i]['features'].clone().detach().to(torch.float32).unsqueeze(0)
        score = torch.sigmoid(model(feat)).item()
        uttids.append(test_df.iloc[i]['uttid'])
        preds.append(score)

results = pd.DataFrame({'uttid': uttids, 'predictions': preds})
results.to_pickle('prediction.pkl')
print(f"Saved {len(results)} predictions to prediction.pkl")
results.head()